# Clase 008 — Funciones: args, kwargs, lambdas, closures

**Parte 0** · Ramalho caps. 7 y 9.

> 🎯 Funciones como first-class objects: callbacks, lambdas, closures — base mental de los decoradores.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
from operator import itemgetter, attrgetter
import time
from functools import wraps

## 1️⃣ Argumentos: las 4 formas

```python
def f(pos, kw='default', *args, kw_only, **kwargs):
    ...
```

- `pos` — posicional o keyword
- `kw='default'` — posicional o keyword con default
- `*args` — captura posicionales restantes en tupla
- `kw_only` — solo se puede pasar nombrado (tras `*args` o `*`)
- `**kwargs` — captura keyword restantes en dict

In [ ]:
def reportar(nombre, edad=0, *extras, ciudad, **meta):
    print(f'nombre  : {nombre}')
    print(f'edad    : {edad}')
    print(f'extras  : {extras}')
    print(f'ciudad  : {ciudad}')
    print(f'meta    : {meta}')

reportar('Ana', 30, 'lectora', 'pianista', ciudad='Madrid', rol='senior', equipo='ML')

## 2️⃣ Keyword-only con `*` separador

```python
def plot(data, *, color='blue', linewidth=1):
    ...  # color y linewidth SOLO se pueden pasar como kwargs

plot(xs, color='red')      # ✅
plot(xs, 'red')            # ❌ TypeError
```

**Por qué útil**: APIs claras. El lector ve `plot(data, color='red', linewidth=2)` y sabe qué hace cada argumento.

## 3️⃣ Funciones como objetos

```python
def saludo(nombre):
    return f'Hola {nombre}'

f = saludo            # asignable
print(f('Mundo'))

fns = [str.upper, str.lower, str.title]  # lista de funciones
for fn in fns:
    print(fn('Hola Mundo'))
```

Esto es lo que hace posible `df.apply(fn)`, `sorted(xs, key=fn)`, `map(fn, xs)`.

In [ ]:
# sorted con key — callback en acción
personas = [
    {'nombre': 'Ana', 'edad': 30},
    {'nombre': 'Bob', 'edad': 25},
    {'nombre': 'Cris', 'edad': 28},
]

# Con lambda
por_edad = sorted(personas, key=lambda p: p['edad'])
print('por edad:', por_edad)

# Con itemgetter (más rápido, más legible para casos simples)
por_nombre = sorted(personas, key=itemgetter('nombre'))
print('por nombre:', por_nombre)

## 4️⃣ Lambdas: dónde sí, dónde no

**Sí**: callback corto, sin nombre relevante:
```python
sorted(xs, key=lambda p: p['edad'])
```

**No**: cuando merece nombre o tiene lógica:
```python
# ❌ ilegible
fn = lambda x: (x*2, x+1) if x > 0 else (0, 0)

# ✅ función con def
def escala_y_offset(x):
    if x > 0:
        return x*2, x+1
    return 0, 0
```

## 5️⃣ Closures — funciones que recuerdan

Una **closure** es una función que captura variables del scope donde fue definida.

```python
def make_counter():
    count = 0                  # variable local de make_counter
    def inner():
        nonlocal count          # le decimos a inner que use la del exterior
        count += 1
        return count
    return inner

contador = make_counter()
contador()  # 1
contador()  # 2
contador()  # 3
```

¿Por qué `count` no muere cuando `make_counter` retorna? Porque `inner` lo capturó y mantiene viva la referencia.

In [ ]:
def make_counter():
    count = 0
    def inner():
        nonlocal count
        count += 1
        return count
    return inner

c1 = make_counter()
c2 = make_counter()   # independiente de c1

print(c1(), c1(), c1())   # 1 2 3
print(c2())               # 1 — su propio count

## 6️⃣ Aplicación: decorador `@memoize` con closure + dict

Un decorador es una función que recibe función y retorna función. Closure + dict = cache.

```python
def memoize(fn):
    cache = {}
    @wraps(fn)
    def wrapper(*args):
        if args not in cache:
            cache[args] = fn(*args)
        return cache[args]
    return wrapper
```

El `cache` vive en el closure → cada llamada con los mismos args devuelve resultado precomputado.

In [ ]:
def memoize(fn):
    cache = {}
    @wraps(fn)
    def wrapper(*args):
        if args not in cache:
            cache[args] = fn(*args)
        return cache[args]
    return wrapper

# Fibonacci recursivo: lento sin memoize
def fib_lento(n):
    if n < 2: return n
    return fib_lento(n-1) + fib_lento(n-2)

@memoize
def fib_rapido(n):
    if n < 2: return n
    return fib_rapido(n-1) + fib_rapido(n-2)

N = 30
t0 = time.perf_counter(); fib_lento(N); t1 = time.perf_counter()
t2 = time.perf_counter(); fib_rapido(N); t3 = time.perf_counter()
print(f'lento  : {(t1-t0)*1000:.1f} ms')
print(f'rápido : {(t3-t2)*1000:.4f} ms')
print(f'speedup: {(t1-t0)/(t3-t2):.0f}x')

## ✅ Checklist

- [ ] Distingo argumentos posicionales, kw, default, *args, **kwargs
- [ ] Sé pasar una función como argumento (callback)
- [ ] Uso lambda solo cuando es corto y claro
- [ ] Entiendo qué es un closure y por qué funciona
- [ ] Implementé un memoize y vi el speedup

## 📝 Homework

Ver `README.md`. `make_counter` explicado, `@memoize` con benchmark Fibonacci, sort por 2 criterios.

## 🔗 Referencias

- Ramalho, *Fluent Python* 2e — caps. 7, 9
- [PEP 3102 keyword-only](https://peps.python.org/pep-3102/)

➡️ **Siguiente:** [009 — Manejo de excepciones y context managers](../009-manejo-de-excepciones-y-context-managers/README.md)